# SNAC API grab (for adding non-connected people)

Using the bones of the original SNAC API script, but much simpler and much smaller scale.

In [1]:
import numpy as np
np.__version__

'2.3.2'

In [2]:
import time
from datetime import datetime
import pandas as pd
import requests
import json

In [3]:
query = """
{
    "command": "read",
    "constellationid": 83834110
}
"""
url = 'https://api.snaccooperative.org/'

In [10]:
# Because we have to find these links manually anyways, I think this is faster than trying to use the API for search.
# Just enter the constellation IDs from SNAC

people = [84648924, 45014216, 73975718, 87859703, 84652230, 29223006, 54706542, 68415070, 85330312]

In [11]:
%%time

people_data = []
attrs = [('dates', 'fromDate', 'birth'), ('dates', 'toDate', 'death'), ('biogHists', 'text', 'bio'), ('nameEntries', 'original', 'name')]

for i in people:
    data = {
            "command": "read",
            "constellationid": i
        }
    query = json.dumps(data)
    response = requests.put(url, query)
    text_json = json.loads(response.text)
    content = text_json['constellation']
    info = {}
    if len(people_data) % 10 == 0:
        print(f'Retrieving record number {len(people_data)+1}')
    for attr in attrs:
        try:
            info[attr[2]] = content[attr[0]][0][attr[1]]
        except:
            info[attr[2]] = ''

    # gender (has different structure so needs to be separate)
    try:
        gender = content['genders'][0]['term']['term']
    except:
        gender = ''
    info['gender'] = gender
            
    # VIAF and ARKID
    
    viaf = ''

    try:
        for i in content['sameAsRelations']:
            if 'viaf' in i['uri']:
                viaf = i['uri']
    except:
        pass
    
    info['arkid'] = content['ark']
    info['viaf'] = viaf
    
    people_data.append(info)

Retrieving record number 1
CPU times: total: 156 ms
Wall time: 13 s


Wall time is super speedy with small batches, recommend creating 10 or so new entries at a time for this reason.

In [12]:
people_data = pd.DataFrame(people_data)

In [13]:
from bs4 import BeautifulSoup

In [14]:
people_data.head(10)

,birth,death,bio,name,gender,arkid,viaf
0,19370721,,<biogHist><p>Distinguished African American jo...,"Britton, John, 1937-",,http://n2t.net/ark:/99166/w6nd6qtk,
1,1935-01-09,,<biogHist><p>Earl G. Graves was raised in the ...,"Graves, Earl G., 1935-",,http://n2t.net/ark:/99166/w63d1xcc,https://viaf.org/viaf/63283367
2,1911-07-02,2007-11-16,"<biogHist>\n <p xmlns=""urn:isbn:1-9...","Rabinowitz, Victor",,http://n2t.net/ark:/99166/w6cd3q6w,https://viaf.org/viaf/8199673
3,1912-03-17,1987-08-24,"Bayard Rustin (b. March 17, 1912, West Chester...","Rustin, Bayard, 1912-1987",,http://n2t.net/ark:/99166/w6fp2049,https://viaf.org/viaf/49401907
4,19380901,,<biogHist><p>Civil rights attorney and law pro...,"Moore, Jane Bond, 1938-",,http://n2t.net/ark:/99166/w67n0zm6,
5,1923-10-12,1988-03-15,,"King C. B. (Chevene Bowers), 1923-1988",,http://n2t.net/ark:/99166/w6gz338s,https://viaf.org/viaf/39155738
6,,,,"Holman, M. Carl.",,http://n2t.net/ark:/99166/w61k8vxg,
7,1939,,"<biogHist><p xmlns=""urn:isbn:1-931666-33-4"">Be...","Brown, Benjamin D., 1939-",,http://n2t.net/ark:/99166/w64k01r6,
8,1935-11-24,2018-07-30,"<p>Ronald Vernie Dellums (November 24, 1935 – ...","Dellums, Ronald V., 1935-2018",Male,http://n2t.net/ark:/99166/w68b1x5j,https://viaf.org/viaf/91321442


In [15]:
def clean_bio(text):
    replace_map = {
        '</p>' : ' ',
        '<span>' : ' ',
        '</span>' : ' '
    }

    if text == '':
        return text

    for k, v in replace_map.items():
        text = text.replace(k, v)

    return BeautifulSoup(text).get_text()

In [16]:
people_data.bio = people_data.bio.apply(clean_bio)
people_data

,birth,death,bio,name,gender,arkid,viaf
0,19370721,,Distinguished African American journalist and ...,"Britton, John, 1937-",,http://n2t.net/ark:/99166/w6nd6qtk,
1,1935-01-09,,Earl G. Graves was raised in the Bedford Stuyv...,"Graves, Earl G., 1935-",,http://n2t.net/ark:/99166/w63d1xcc,https://viaf.org/viaf/63283367
2,1911-07-02,2007-11-16,\nVictor Rabinowitz was the son of Jewish immi...,"Rabinowitz, Victor",,http://n2t.net/ark:/99166/w6cd3q6w,https://viaf.org/viaf/8199673
3,1912-03-17,1987-08-24,"Bayard Rustin (b. March 17, 1912, West Chester...","Rustin, Bayard, 1912-1987",,http://n2t.net/ark:/99166/w6fp2049,https://viaf.org/viaf/49401907
4,19380901,,Civil rights attorney and law professor Jane B...,"Moore, Jane Bond, 1938-",,http://n2t.net/ark:/99166/w67n0zm6,
5,1923-10-12,1988-03-15,,"King C. B. (Chevene Bowers), 1923-1988",,http://n2t.net/ark:/99166/w6gz338s,https://viaf.org/viaf/39155738
6,,,,"Holman, M. Carl.",,http://n2t.net/ark:/99166/w61k8vxg,
7,1939,,"Benjamin D. Brown, lawyer and politician, Geor...","Brown, Benjamin D., 1939-",,http://n2t.net/ark:/99166/w64k01r6,
8,1935-11-24,2018-07-30,"Ronald Vernie Dellums (November 24, 1935 – Jul...","Dellums, Ronald V., 1935-2018",Male,http://n2t.net/ark:/99166/w68b1x5j,https://viaf.org/viaf/91321442


In [17]:
people_data.gender.value_counts()

gender
        8
Male    1
Name: count, dtype: int64

In [18]:
people_data.viaf.value_counts()['']

np.int64(4)

In [19]:
people_data.bio.value_counts()['']

np.int64(2)

In [20]:
people_data.death.value_counts()['']

np.int64(5)

In [21]:
people_data.to_csv('SNAC_people_update.csv')

#### Reading back in - don't want to have to retrieve all the data anew

In [22]:
people = pd.read_csv('SNAC_people_update.csv')

In [23]:
people = people.drop('Unnamed: 0', axis = 1)

In [24]:
people.birth.unique()

array(['19370721', '1935-01-09', '1911-07-02', '1912-03-17', '19380901',
       '1923-10-12', nan, '1939', '1935-11-24'], dtype=object)

In [25]:
def parse_birthdate(date):
    try:
        date = pd.to_datetime(date, format='%Y-%m-%d', errors='raise').date()
        return date, date
    except ValueError:
        pass
    
    try:
        year = pd.to_datetime(date, format='%Y', errors='raise').year
        start = pd.to_datetime(f'{year}-01-01').date()
        end   = pd.to_datetime(f'{year}-12-31').date()
        return start, end
    except ValueError:
        pass
    
    return None, None

# date_ranges = [parse_birthdate(v) for v in people_test.birth]

In [28]:
people[['birth_start', 'birth_end']] = people.birth.apply(parse_birthdate).apply(pd.Series)
people[['death_start', 'death_end']] = people.death.apply(parse_birthdate).apply(pd.Series)
people = people.drop(['birth', 'death'], axis = 1)
people

,bio,name,gender,arkid,viaf,birth_start,birth_end,death_start,death_end
0,Distinguished African American journalist and ...,"Britton, John, 1937-",NaN,http://n2t.net/ark:/99166/w6nd6qtk,NaN,None,None,None,None
1,Earl G. Graves was raised in the Bedford Stuyv...,"Graves, Earl G., 1935-",NaN,http://n2t.net/ark:/99166/w63d1xcc,https://viaf.org/viaf/63283367,1935-01-09,1935-01-09,None,None
2,\nVictor Rabinowitz was the son of Jewish immi...,"Rabinowitz, Victor",NaN,http://n2t.net/ark:/99166/w6cd3q6w,https://viaf.org/viaf/8199673,1911-07-02,1911-07-02,2007-11-16,2007-11-16
3,"Bayard Rustin (b. March 17, 1912, West Chester...","Rustin, Bayard, 1912-1987",NaN,http://n2t.net/ark:/99166/w6fp2049,https://viaf.org/viaf/49401907,1912-03-17,1912-03-17,1987-08-24,1987-08-24
4,Civil rights attorney and law professor Jane B...,"Moore, Jane Bond, 1938-",NaN,http://n2t.net/ark:/99166/w67n0zm6,NaN,None,None,None,None
5,NaN,"King C. B. (Chevene Bowers), 1923-1988",NaN,http://n2t.net/ark:/99166/w6gz338s,https://viaf.org/viaf/39155738,1923-10-12,1923-10-12,1988-03-15,1988-03-15
6,NaN,"Holman, M. Carl.",NaN,http://n2t.net/ark:/99166/w61k8vxg,NaN,None,None,None,None
7,"Benjamin D. Brown, lawyer and politician, Geor...","Brown, Benjamin D., 1939-",NaN,http://n2t.net/ark:/99166/w64k01r6,NaN,1939-01-01,1939-12-31,None,None
8,"Ronald Vernie Dellums (November 24, 1935 – Jul...","Dellums, Ronald V., 1935-2018",Male,http://n2t.net/ark:/99166/w68b1x5j,https://viaf.org/viaf/91321442,1935-11-24,1935-11-24,2018-07-30,2018-07-30


In [29]:
people = people.rename(columns = {'bio':'description', 'name':'title'})
people['arkid_label'] = 'SNAC Record'
people['viaf_label'] = 'VIAF Record'
people.head()

,description,title,gender,arkid,viaf,birth_start,birth_end,death_start,death_end,arkid_label,viaf_label
0,Distinguished African American journalist and ...,"Britton, John, 1937-",NaN,http://n2t.net/ark:/99166/w6nd6qtk,NaN,None,None,None,None,SNAC Record,VIAF Record
1,Earl G. Graves was raised in the Bedford Stuyv...,"Graves, Earl G., 1935-",NaN,http://n2t.net/ark:/99166/w63d1xcc,https://viaf.org/viaf/63283367,1935-01-09,1935-01-09,None,None,SNAC Record,VIAF Record
2,\nVictor Rabinowitz was the son of Jewish immi...,"Rabinowitz, Victor",NaN,http://n2t.net/ark:/99166/w6cd3q6w,https://viaf.org/viaf/8199673,1911-07-02,1911-07-02,2007-11-16,2007-11-16,SNAC Record,VIAF Record
3,"Bayard Rustin (b. March 17, 1912, West Chester...","Rustin, Bayard, 1912-1987",NaN,http://n2t.net/ark:/99166/w6fp2049,https://viaf.org/viaf/49401907,1912-03-17,1912-03-17,1987-08-24,1987-08-24,SNAC Record,VIAF Record
4,Civil rights attorney and law professor Jane B...,"Moore, Jane Bond, 1938-",NaN,http://n2t.net/ark:/99166/w67n0zm6,NaN,None,None,None,None,SNAC Record,VIAF Record


In [30]:
people = people.replace('nan', None)

In [32]:
type(people['description'][1])

str

In [33]:
people.description = people.description.apply(lambda x: f'From Social Networks and Archival Context: "{x}"' if pd.notna(x) else None)
people

,description,title,gender,arkid,viaf,birth_start,birth_end,death_start,death_end,arkid_label,viaf_label
0,"From Social Networks and Archival context: ""Di...","Britton, John, 1937-",NaN,http://n2t.net/ark:/99166/w6nd6qtk,NaN,None,None,None,None,SNAC Record,VIAF Record
1,"From Social Networks and Archival context: ""Ea...","Graves, Earl G., 1935-",NaN,http://n2t.net/ark:/99166/w63d1xcc,https://viaf.org/viaf/63283367,1935-01-09,1935-01-09,None,None,SNAC Record,VIAF Record
2,"From Social Networks and Archival context: ""\n...","Rabinowitz, Victor",NaN,http://n2t.net/ark:/99166/w6cd3q6w,https://viaf.org/viaf/8199673,1911-07-02,1911-07-02,2007-11-16,2007-11-16,SNAC Record,VIAF Record
3,"From Social Networks and Archival context: ""Ba...","Rustin, Bayard, 1912-1987",NaN,http://n2t.net/ark:/99166/w6fp2049,https://viaf.org/viaf/49401907,1912-03-17,1912-03-17,1987-08-24,1987-08-24,SNAC Record,VIAF Record
4,"From Social Networks and Archival context: ""Ci...","Moore, Jane Bond, 1938-",NaN,http://n2t.net/ark:/99166/w67n0zm6,NaN,None,None,None,None,SNAC Record,VIAF Record
5,None,"King C. B. (Chevene Bowers), 1923-1988",NaN,http://n2t.net/ark:/99166/w6gz338s,https://viaf.org/viaf/39155738,1923-10-12,1923-10-12,1988-03-15,1988-03-15,SNAC Record,VIAF Record
6,None,"Holman, M. Carl.",NaN,http://n2t.net/ark:/99166/w61k8vxg,NaN,None,None,None,None,SNAC Record,VIAF Record
7,"From Social Networks and Archival context: ""Be...","Brown, Benjamin D., 1939-",NaN,http://n2t.net/ark:/99166/w64k01r6,NaN,1939-01-01,1939-12-31,None,None,SNAC Record,VIAF Record
8,"From Social Networks and Archival context: ""Ro...","Dellums, Ronald V., 1935-2018",Male,http://n2t.net/ark:/99166/w68b1x5j,https://viaf.org/viaf/91321442,1935-11-24,1935-11-24,2018-07-30,2018-07-30,SNAC Record,VIAF Record


In [34]:
people.to_csv('people_new.csv', index = False)